# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata as an object and print summary
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Identifier: {meta.identifier}")
print(f"Citation: {meta.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All dataset entities (record sets, fields, columns etc.) are referenced and listed by their `@id` fields.

In [ ]:
# Get the list of record sets via their @id
record_sets = []
if hasattr(meta, 'recordSet') and meta.recordSet:
    record_sets = [r['@id'] if isinstance(r, dict) else r for r in meta.recordSet]
else:
    # Fallback: Find record sets from the Croissant document directly
    # mlcroissant exposes metadata as an object; but it may have no recordSet loaded
    # Instead, read raw JSON-LD and extract record sets
    import requests, json
    croissant_schema = requests.get(croissant_url).json()
    record_sets = [rs['@id'] for rs in croissant_schema.get('recordSet', [])]
    # In case recordSet is not at root, search for all nodes with type 'RecordSet'
    if not record_sets:
        for x in croissant_schema.values() if isinstance(croissant_schema, dict) else []:
            if isinstance(x, dict) and x.get('@type') == 'RecordSet' and '@id' in x:
                record_sets.append(x['@id'])

print("Available Record Sets by @id:")
for rsid in record_sets:
    print(f"- {rsid}")

In [ ]:
# List all fields for each record set by their @id
croissant_schema = requests.get(croissant_url).json()
print("\nFields for each record set:")
for rsid in record_sets:
    record_set_obj = None
    for obj in croissant_schema.get('recordSet', []):
        if obj['@id'] == rsid:
            record_set_obj = obj
            break
    if record_set_obj:
        fields = record_set_obj.get('field', [] if 'field' not in record_set_obj else [])
        fields_ids = [f['@id'] if isinstance(f, dict) else f for f in fields]
        print(f"Record set @id: {rsid}")
        for fid in fields_ids:
            print(f"  - Field @id: {fid}")
    else:
        print(f"Record set @id: {rsid} (unable to locate fields)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# For demonstration, use the first record set
dataframes = {}

for rsid in record_sets:
    # Load records for each record set by @id
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Columns for record set @id '{rsid}': {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records loaded for {rsid}")
    except Exception as e:
        print(f"Error loading records for {rsid}: {e}")

# For summary, pick first record set as primary
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id and primary_record_set_id in dataframes:
    print(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section applies operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis, e.g. GAD-7 Score
# These are identified by their @id in the Croissant schema
# Example @id for GAD-7: 'http://senscience.ai/gad7_total_score' (actual ID may differ)
numeric_field_id = None

# Find a numeric field ID in the primary record set
primary_fields = []
for obj in croissant_schema.get('recordSet', []):
    if obj['@id'] == primary_record_set_id:
        primary_fields = obj.get('field', [])
        break

# Find first numeric-type field
numeric_fields = []
for f in primary_fields:
    field_obj = f if isinstance(f, dict) else None
    if field_obj:
        dtype = field_obj.get('dataType', '')
        if dtype in ['schema:Float', 'schema:Integer']:
            numeric_fields.append(field_obj['@id'])

if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # Fallback: Use column names heuristically
    for col in dataframes[primary_record_set_id].columns:
        if 'score' in col.lower() or 'age' in col.lower():
            numeric_field_id = col
            break

print(f"Using numeric field @id: {numeric_field_id}")

# Filtering based on threshold for this field
threshold = 10
if numeric_field_id and numeric_field_id in dataframes[primary_record_set_id].columns:
    filtered_df = dataframes[primary_record_set_id][dataframes[primary_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by another field, e.g. education level
    group_field_id = None
    for col in dataframes[primary_record_set_id].columns:
        if 'education' in col.lower():
            group_field_id = col
            break
    print(f"Grouping by field @id: {group_field_id}")
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found or no numeric fields available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization: Histogram and scatter plot between two numeric fields
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in dataframes[primary_record_set_id].columns:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[primary_record_set_id][numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot vs another numeric field, e.g. PHQ-9
    second_numeric_field = None
    for col in dataframes[primary_record_set_id].columns:
        if col != numeric_field_id and 'phq' in col.lower():
            second_numeric_field = col
            break
    if second_numeric_field:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(
            x=dataframes[primary_record_set_id][numeric_field_id],
            y=dataframes[primary_record_set_id][second_numeric_field]
        )
        plt.title(f"{numeric_field_id} vs {second_numeric_field}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(second_numeric_field)
        plt.show()
    else:
        print("No second numeric field (like PHQ-9) found for scatterplot.")
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- The dataset was loaded using the `mlcroissant` library and referenced entities by their `@id`.
- Main record sets and fields were identified via the Croissant schema.
- Numeric survey scores (e.g., GAD-7) were analyzed for distributions and filtered for high scores.
- Data was grouped and visualized to show relationships between survey scores and education levels.

This notebook provides a reproducible path for exploring complex datasets defined by Croissant metadata using entity IDs for robust data handling.